In [1]:
!pip install kagglehub

In [2]:
import os
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import classification_report, confusion_matrix

In [3]:
print("Downloading dataset...")

dataset_path = kagglehub.dataset_download("emmarex/plantdisease")

print("Dataset downloaded!")
print("Dataset path:", dataset_path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Dataset downloaded!
Dataset path: /kaggle/input/plantdisease


In [4]:
data_dir = None

for folder in os.listdir(dataset_path):

    full_path = os.path.join(dataset_path, folder)

    if os.path.isdir(full_path):
        data_dir = full_path
        break

print("Using dataset folder:", data_dir)

Using dataset folder: /kaggle/input/plantdisease/PlantVillage


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [6]:
transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

class_names = dataset.classes

print("Number of classes:", len(class_names))
print("Classes:", class_names)

Number of classes: 15
Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [8]:
total_size = len(dataset)

train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 14446
Validation size: 3095
Test size: 3097


In [9]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [10]:
class BaselineCNN(nn.Module):

    def __init__(self, num_classes):

        super(BaselineCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [11]:
model = BaselineCNN(
    num_classes=len(class_names)
).to(device)

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=15, bias=True)
  )
)


In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [13]:
epochs = 10

print("Starting training...\n")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {running_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%\n")

print("Training finished!")

Starting training...

Epoch [1/10]
Loss: 585.3210
Training Accuracy: 58.05%

Epoch [2/10]
Loss: 293.3303
Training Accuracy: 79.00%

Epoch [3/10]
Loss: 214.0157
Training Accuracy: 83.93%

Epoch [4/10]
Loss: 167.8465
Training Accuracy: 87.62%

Epoch [5/10]
Loss: 129.4118
Training Accuracy: 90.27%

Epoch [6/10]
Loss: 108.9364
Training Accuracy: 92.16%

Epoch [7/10]
Loss: 93.2585
Training Accuracy: 93.15%

Epoch [8/10]
Loss: 80.2586
Training Accuracy: 94.03%

Epoch [9/10]
Loss: 61.7613
Training Accuracy: 95.53%

Epoch [10/10]
Loss: 71.3081
Training Accuracy: 94.79%

Training finished!


In [14]:
torch.save(model.state_dict(), "baseline_cnn.pth")

print("Model saved!")

Model saved!


In [15]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predicted.cpu().numpy())

In [16]:
print("Classification Report:\n")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

Classification Report:

                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.97      0.93      0.95       146
                     Pepper__bell___healthy       0.96      0.98      0.97       228
                      Potato___Early_blight       0.96      0.97      0.96       155
                       Potato___Late_blight       0.88      0.93      0.91       146
                           Potato___healthy       0.93      0.68      0.79        19
                      Tomato_Bacterial_spot       0.97      0.94      0.96       318
                        Tomato_Early_blight       0.88      0.80      0.84       142
                         Tomato_Late_blight       0.90      0.92      0.91       283
                           Tomato_Leaf_Mold       0.96      0.89      0.92       127
                  Tomato_Septoria_leaf_spot       0.93      0.90      0.91       273
Tomato_Spider_mites_Two_spotted_spider_m

In [17]:
print("Confusion Matrix:\n")

print(confusion_matrix(
    all_labels,
    all_predictions
))

Confusion Matrix:

[[136   2   0   0   0   0   2   1   0   3   0   1   1   0   0]
 [  2 224   0   0   1   0   0   0   0   0   0   1   0   0   0]
 [  0   0 150   3   0   0   0   2   0   0   0   0   0   0   0]
 [  1   0   0 136   0   1   1   5   0   1   0   1   0   0   0]
 [  0   2   0   1  13   0   0   1   0   1   0   1   0   0   0]
 [  0   1   0   2   0 300   0   1   0   0   0   0  14   0   0]
 [  0   0   0   0   0   6 114  12   0   2   1   6   1   0   0]
 [  0   1   2  10   0   0   6 261   1   0   0   0   2   0   0]
 [  0   0   0   0   0   0   0   2 113   9   2   1   0   0   0]
 [  1   3   4   1   0   2   2   5   2 245   2   3   2   1   0]
 [  0   0   0   0   0   0   3   0   2   0 237   8   0   1   0]
 [  0   0   0   1   0   0   1   0   0   1   7 194   1   0   0]
 [  0   0   0   0   0   1   0   0   0   0   6   0 495   0   0]
 [  0   0   0   0   0   0   1   0   0   1   0   0   0  51   0]
 [  0   0   0   0   0   0   0   0   0   0   1   1   0   0 247]]
